In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

Base sem quali

In [2]:
df_sem_quali = pd.read_csv(r"D:\pedro\Documents\modelo_f1\DATA\oficial.csv")
df_sem_quali.drop(columns=["posicao_quali_atual", "q1_atual", "q2_atual", "q3_atual", "dif_para_pole_atual"], inplace=True)
df_sem_quali.head()

,temporada_atual,rodada_atual,id_piloto_atual,target,id_circuito_atual,id_equipe_atual,grid_anterior,posicao_ultima_corrida,status,pontos_anterior_individual,...,media_ultimas_3_anterior,media_ultimas_5_anterior,qtde_abandonos_anterior,media_posicao_ganha_anterior,tendencia_desempenho,temp_ar_media,temp_pista_media,umidade_media,corrida_molhada,perc_voltas_chuva
0,2018,1,alonso,5,albert_park,mclaren,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,24.08,36.32,30.92,0,4.5
1,2018,1,bottas,8,albert_park,mercedes,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,24.08,36.32,30.92,0,4.5
2,2018,1,brendon_hartley,15,albert_park,toro_rosso,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,24.08,36.32,30.92,0,4.5
3,2018,1,ericsson,19,albert_park,sauber,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,24.08,36.32,30.92,0,4.5
4,2018,1,gasly,18,albert_park,toro_rosso,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,24.08,36.32,30.92,0,4.5


In [3]:
df_sem_quali["target"] = (df_sem_quali["target"] <= 3).astype(int)

In [4]:
df_sem_quali["posicao_equipe_anterior"] = pd.to_numeric(df_sem_quali["posicao_equipe_anterior"], errors="coerce")

df_sem_quali.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3612 entries, 0 to 3611
Data columns (total 26 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   temporada_atual               3612 non-null   int64  
 1   rodada_atual                  3612 non-null   int64  
 2   id_piloto_atual               3612 non-null   object 
 3   target                        3612 non-null   int32  
 4   id_circuito_atual             3612 non-null   object 
 5   id_equipe_atual               3612 non-null   object 
 6   grid_anterior                 3403 non-null   float64
 7   posicao_ultima_corrida        3403 non-null   float64
 8   status                        3403 non-null   object 
 9   pontos_anterior_individual    3403 non-null   float64
 10  posicao_equipe_anterior       3426 non-null   float64
 11  pontos_equipe_anterior        3430 non-null   float64
 12  vitorias_equipe_anterior      3430 non-null   float64
 13  pon

In [5]:
df_sem_quali.describe()

,temporada_atual,rodada_atual,target,grid_anterior,posicao_ultima_corrida,pontos_anterior_individual,posicao_equipe_anterior,pontos_equipe_anterior,vitorias_equipe_anterior,pontos_anterior,...,media_ultimas_3_anterior,media_ultimas_5_anterior,qtde_abandonos_anterior,media_posicao_ganha_anterior,tendencia_desempenho,temp_ar_media,temp_pista_media,umidade_media,corrida_molhada,perc_voltas_chuva
count,3612.000000,3612.000000,3612.000000,3403.000000,3403.000000,3403.000000,3426.000000,3430.000000,3430.000000,3417.000000,...,3417.000000,3417.000000,3417.000000,3417.000000,3417.000000,3612.000000,3612.000000,3612.000000,3612.000000,3612.000000
mean,2021.831395,11.097453,0.149502,10.272113,10.514840,5.070967,5.512843,112.068222,1.062391,56.075505,...,10.518557,10.517504,0.546093,-0.238654,0.001054,23.440631,35.365133,53.623560,0.033223,5.357032
std,2.430326,6.445127,0.356632,5.823335,5.790889,7.215496,2.884033,151.755111,2.681751,79.601597,...,4.659359,4.428662,1.024151,3.338725,1.472039,4.925805,9.122735,16.604338,0.179242,16.279797
min,2018.000000,1.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,...,1.000000,1.000000,0.000000,-18.000000,-6.800000,9.500000,15.210000,7.110000,0.000000,0.000000
25%,2020.000000,5.000000,0.000000,5.000000,5.000000,0.000000,3.000000,12.000000,0.000000,4.000000,...,7.000000,7.000000,0.000000,-2.000000,-0.730000,20.120000,28.970000,42.320000,0.000000,0.000000
50%,2022.000000,11.000000,0.000000,10.000000,11.000000,0.000000,6.000000,46.000000,0.000000,22.000000,...,11.000000,11.200000,0.000000,0.000000,0.000000,23.480000,35.375000,54.720000,0.000000,0.000000
75%,2024.000000,16.000000,0.000000,15.000000,16.000000,9.000000,8.000000,148.000000,0.000000,71.000000,...,14.330000,14.000000,1.000000,2.000000,0.750000,27.130000,42.030000,63.900000,0.000000,0.000000
max,2026.000000,24.000000,1.000000,22.000000,22.000000,26.000000,11.000000,822.000000,20.000000,549.000000,...,22.000000,22.000000,7.000000,14.000000,5.530000,36.570000,54.530000,96.440000,1.000000,98.000000


In [6]:
df_sem_quali.drop(columns=["id_piloto_atual"], inplace=True)

In [7]:
X = df_sem_quali.drop(columns=['target'])
y = df_sem_quali['target']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

In [9]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [10]:
onehotencoder = ColumnTransformer(transformers=[
    ('OneHot', OneHotEncoder(handle_unknown='ignore'), ['id_circuito_atual', 'id_equipe_atual', 'status'])
], remainder='passthrough')

In [11]:
X_train_transformed = onehotencoder.fit_transform(X_train)
X_test_transformed = onehotencoder.transform(X_test)

In [12]:
param_grid = {
    "max_depth": [3, 4, 5],
    "learning_rate": [0.01, 0.03, 0.05],
    "n_estimators": [300, 500, 700, 1000],
    "scale_pos_weight": [2, 3],       
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.6, 0.7, 0.8], 
    "min_child_weight": [5, 7, 10],     
    "gamma": [0.2, 0.4, 0.6],          
    "reg_alpha": [0, 0.1, 0.5],       
    "reg_lambda": [1, 1.5, 2],       
}

xgb_model = xgb.XGBClassifier(random_state=42)

grid_search = RandomizedSearchCV(xgb_model, param_grid, cv=10, scoring="accuracy", n_iter=100, n_jobs=-1, verbose=2, random_state=42)
grid_search.fit(X_train_transformed, y_train)

best_xgb = grid_search.best_estimator_

print("Melhores parâmetros:", grid_search.best_params_)
print("Melhor acurácia:", grid_search.best_score_)

Fitting 10 folds for each of 100 candidates, totalling 1000 fits
Melhores parâmetros: {'subsample': 0.8, 'scale_pos_weight': 2, 'reg_lambda': 2, 'reg_alpha': 0, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 5, 'learning_rate': 0.03, 'gamma': 0.4, 'colsample_bytree': 0.8}
Melhor acurácia: 0.8712274125336409


In [13]:
y_pred = best_xgb.predict(X_test_transformed)

test_accuracy = accuracy_score(y_test, y_pred)
print("Acurácia: ", test_accuracy)

print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nMatriz de confusão:\n", confusion_matrix(y_test, y_pred))

Acurácia:  0.8658367911479945

Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.92      0.92       616
           1       0.54      0.57      0.56       107

    accuracy                           0.87       723
   macro avg       0.73      0.74      0.74       723
weighted avg       0.87      0.87      0.87       723


Matriz de confusão:
 [[565  51]
 [ 46  61]]


Entendendo as features utilizadas

In [ ]:
colunas_ohe = onehotencoder.named_transformers_['OneHot'].get_feature_names_out(['id_circuito_atual', 'id_equipe_atual', 'status'])
colunas_resto = [c for c in X.columns if c not in ['id_circuito_atual', 'id_equipe_atual']]
feature_names = list(colunas_ohe) + list(colunas_resto)

feature_importances = best_xgb.feature_importances_
sorted_indices = np.argsort(feature_importances)[::-1]

plt.figure(figsize=(14, 5))
plt.bar(range(len(feature_importances)), feature_importances[sorted_indices], align="center")
plt.xticks(range(len(feature_importances)), np.array(feature_names)[sorted_indices], rotation=90)
plt.xlabel("Importância da Feature")
plt.title("XGB Importância da Feature para o DF da F1")
plt.tight_layout()
plt.show()

In [ ]:
import joblib

joblib.dump(best_xgb, r"D:\pedro\Documents\modelo_f1\modelos\modelo_sem_quali.pkl")
joblib.dump(onehotencoder, r"D:\pedro\Documents\modelo_f1\modelos\encoder_sem_quali.pkl")

# Testes

In [3]:
import joblib
import pandas as pd

modelo = joblib.load(r"D:\pedro\Documents\modelo_f1\modelos\modelo_sem_quali.pkl")
encoder = joblib.load(r"D:\pedro\Documents\modelo_f1\modelos\encoder_sem_quali.pkl")

df = pd.read_csv(r"D:\pedro\Documents\modelo_f1\DATA\a.csv")

X_train_transformed = encoder.transform(df)
predicoes = modelo.predict_proba(X_train_transformed)

print(predicoes)

[[0.53330976 0.46669024]
 [0.80553794 0.19446206]
 [0.58441913 0.41558087]
 [0.8845336  0.11546642]
 [0.5863427  0.41365728]
 [0.9749756  0.02502443]
 [0.98119414 0.01880584]
 [0.9793893  0.02061068]
 [0.9480714  0.0519286 ]
 [0.9677106  0.03228939]
 [0.99372494 0.00627504]
 [0.9933134  0.00668664]
 [0.99157286 0.00842717]
 [0.9977092  0.00229079]
 [0.80807686 0.19192317]
 [0.19950408 0.8004959 ]
 [0.9857652  0.0142348 ]
 [0.995454   0.00454597]
 [0.99650294 0.00349705]
 [0.9839064  0.01609364]
 [0.9964133  0.00358668]
 [0.9962966  0.00370341]]


In [4]:
probs = modelo.predict_proba(X_train_transformed)[:, 1]

df_result = df.copy()
df_result['prob_podio'] = probs
df_result['predicao'] = modelo.predict(X_train_transformed)

print(df_result[['id_piloto_atual', 'prob_podio', 'predicao']]
      .sort_values('prob_podio', ascending=False))

   id_piloto_atual  prob_podio  predicao
15       antonelli    0.800496         1
0         hamilton    0.466690         0
2           norris    0.415581         0
4          piastri    0.413657         0
1          russell    0.194462         0
14         leclerc    0.191923         0
3   max_verstappen    0.115466         0
8   arvid_lindblad    0.051929         0
9        colapinto    0.032289         0
5           hadjar    0.025024         0
7           lawson    0.020611         0
6            gasly    0.018806         0
19      hulkenberg    0.016094         0
16         bearman    0.014235         0
12            ocon    0.008427         0
11           sainz    0.006687         0
10       bortoleto    0.006275         0
17           albon    0.004546         0
21          stroll    0.003703         0
20          bottas    0.003587         0
18          alonso    0.003497         0
13           perez    0.002291         0


In [ ]:
import os
import streamlit as st
from google.cloud import storage
from google.oauth2 import service_account
import pickle

def salva_gcs(caminho_gcs, caminho_local):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "D:\pedro\Documents\modelo_f1\clever-axe-457319-g8-833d2d4ab67f.json"

    client = client = storage.Client()
    bucket = client.bucket("f1-dashboard-pilotos")
    blob = bucket.blob(caminho_gcs)

    blob.upload_from_filename(caminho_local)

modelos = [
    ("modelos/modelo_sem_quali.pkl",  r"D:\pedro\Documents\modelo_f1\modelos\modelo_sem_quali.pkl"),
    ("modelos/encoder_sem_quali.pkl", r"D:\pedro\Documents\modelo_f1\modelos\encoder_sem_quali.pkl"),
]
for caminho_gcs, caminho_local in modelos:
    salva_gcs(caminho_gcs, caminho_local)

In [ ]:
from sklearn.metrics import classification_report
import json
import os

metricas = classification_report(
    y_test,
    y_pred,
    output_dict=True
)

metricas_classe_1 = metricas["1"]
metricas_classe_1["accuracy"] = test_accuracy

def salva_gcs(dados):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "D:\pedro\Documents\modelo_f1\clever-axe-457319-g8-833d2d4ab67f.json"

    client = client = storage.Client()
    bucket = client.bucket("f1-dashboard-pilotos")
    blob = bucket.blob("metricas_sem_quali.json")
    blob.upload_from_string(
        json.dumps(dados), 
        content_type="application/json"
    )

salva_gcs(metricas_classe_1)